# Capstone 02: BookFinder Starter
**BookFinder: Library Book Finder with Natural Language Search**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
This notebook generates the project data, shows a working baseline, and seeds your golden set. Beat the baseline in your project.

* **Option A (Google Colab)** cells are active; **Option B (VS Code)** lines are commented. Toggle with `Ctrl + /`.

## 1. Setup

In [ ]:
# ===== Option A: Google Colab (active) =====
# Add your key once: click the key icon (Secrets) in the left sidebar, name it OPENAI_API_KEY, and allow notebook access.
!pip install -q openai scikit-learn numpy pandas rank_bm25

# ===== Option B: VS Code / local =====
# pip install openai scikit-learn numpy pandas rank_bm25
# Create a .env file or set the variable in your terminal:  OPENAI_API_KEY=... (git-ignored .env file)
# No key? Everything still runs in OFFLINE demo mode with a tiny built-in stand-in model.
print("Setup done")

In [ ]:
%%writefile genai_helpers.py
"""Shared helpers for the GenAI days.
llm()   -> calls OpenAI when a key is available, otherwise a small OFFLINE stand-in so every cell still runs.
embed() -> OpenAI embeddings online, a hashing-based stand-in offline (same API, lower quality).
"""
import json, os, re
import numpy as np

def _get_key():
    if os.getenv("OFFLINE") == "1":                 # force offline demo mode (used by tools/check_notebooks.py)
        return None
    try:
        from google.colab import userdata          # Colab Secrets
        return userdata.get("OPENAI_API_KEY")
    except Exception:
        return os.getenv("OPENAI_API_KEY")

OPENAI_API_KEY = _get_key()
ONLINE = bool(OPENAI_API_KEY)
# Default model: confirm against the approved model list (docs/MODEL_GUIDE.md). Override with LLM_MODEL.
MODEL = os.getenv("LLM_MODEL", "gpt-6.1-sol")
EMBED_MODEL = os.getenv("EMBED_MODEL", "text-embedding-3-small")
if ONLINE:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

WORD = re.compile(r"[a-z0-9]+")

def _words(t):
    # crude normalisation so "expires" matches "expire"
    return {w[:-1] if w.endswith("s") and len(w) > 4 else w for w in WORD.findall(t.lower()) if len(w) > 3}

def offline_llm(prompt, system=""):
    """A deliberately simple stand-in. It is NOT intelligent; it lets the code paths run without a key."""
    low = prompt.lower()
    if "context:" in low and "question:" in low:
        ctx = re.split(r"(?i)context:", prompt, 1)[1]
        ctx, q = re.split(r"(?i)question:", ctx, 1)
        sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", ctx.strip()) if s.strip()]
        best = max(sents, key=lambda s: len(_words(s) & _words(q)), default="")
        return best if best and _words(best) & _words(q) else "I don't know based on the provided context."
    if "classify" in low and "positive" in low:
        neg = sum(w in low for w in ["slow", "bad", "broken", "late", "angry", "worst", "refund"])
        pos = sum(w in low for w in ["great", "love", "fast", "thanks", "excellent", "happy"])
        return "negative" if neg > pos else "positive" if pos > neg else "neutral"
    if "json" in low:
        text = prompt.strip().splitlines()[-1]
        return json.dumps({"summary": " ".join(text.split()[:12]), "offline": True})
    first = re.split(r"(?<=[.!?])\s+", prompt.strip())[0]
    return f"[offline demo] {first[:160]}"

def llm(prompt, system="You are a helpful assistant.", temperature=0.2, model=None):
    if ONLINE:
        r = client.chat.completions.create(
            model=model or MODEL, temperature=temperature,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}])
        return r.choices[0].message.content
    return offline_llm(prompt, system)

def embed(texts):
    """Return an (n, d) array of unit-length vectors."""
    if isinstance(texts, str):
        texts = [texts]
    if ONLINE:
        r = client.embeddings.create(model=EMBED_MODEL, input=texts)
        v = np.array([d.embedding for d in r.data])
    else:
        from sklearn.feature_extraction.text import HashingVectorizer
        hv = HashingVectorizer(n_features=1024, ngram_range=(1, 2), alternate_sign=False, norm=None)
        v = hv.transform([t.lower() for t in texts]).toarray()
    n = np.linalg.norm(v, axis=1, keepdims=True)
    return v / np.where(n == 0, 1, n)

print("Mode:", f"ONLINE ({MODEL})" if ONLINE else "OFFLINE demo (add OPENAI_API_KEY to use a real model)")

In [ ]:
import importlib, genai_helpers
importlib.reload(genai_helpers)
from genai_helpers import llm, embed, ONLINE

## 2. Generate the catalogue, copies and loans

In [ ]:
import numpy as np, pandas as pd, json, os, re, sqlite3
rng = np.random.default_rng(202)
os.makedirs("data", exist_ok=True)
subjects = {"mystery": ["detective", "secret", "case"], "historical fiction": ["samurai", "empire", "war"], "children's science": ["space", "planets", "robots"],
            "cooking": ["recipes", "spices", "ramen"], "travel": ["Kyoto", "Kerala", "railway"], "self-help": ["habits", "focus", "calm"]}
places = ["Kyoto", "Osaka", "Bengaluru", "Mysuru", "Tokyo", "Goa"]
books = []
for i in range(8000):
    subj = rng.choice(list(subjects)); w = rng.choice(subjects[subj]); place = rng.choice(places)
    lang = "ja" if rng.random() < 0.3 else "en"
    books.append({"book_id": f"B{i:05d}", "title": f"The {w.title()} of {place}", "author": f"{rng.choice(['Ito', 'Rao', 'Sato', 'Iyer', 'Mori', 'Nair'])} {rng.choice(['K.', 'A.', 'M.', 'S.'])}",
                  "language": lang, "subject": subj, "year": int(rng.integers(1960, 2026)),
                  "summary": f"A {rng.choice(['gentle', 'thrilling', 'warm', 'thoughtful'])} {subj} book about {w} in {place}."})
books = pd.DataFrame(books)
copies = pd.DataFrame([{"copy_id": f"C{i:06d}", "book_id": b, "branch": rng.choice(["BLR-1", "BLR-2", "TYO-1", "TYO-2", "OSA-1"]), "available": bool(rng.random() < 0.6)}
                       for i, b in enumerate(np.repeat(books.book_id, 2))])
books.to_csv("data/books.csv", index=False); copies.to_csv("data/copies.csv", index=False)
print(books.shape, copies.shape); print(books.head(3))

## 3. Baseline search: keyword, embeddings and hybrid (RRF)

In [ ]:
from rank_bm25 import BM25Okapi
docs = (books.title + " " + books.subject + " " + books.summary).tolist()
tok = lambda s: re.findall(r"[a-z0-9']+", s.lower())
bm25 = BM25Okapi([tok(d) for d in docs])
vecs = embed(docs)
def kw(q, k=50): return list(np.argsort(bm25.get_scores(tok(q)))[::-1][:k])
def vec(q, k=50): return list(np.argsort(vecs @ embed(q)[0])[::-1][:k])
def rrf(lists, k=60):
    s = {}
    for l in lists:
        for r, d in enumerate(l, 1): s[d] = s.get(d, 0) + 1 / (k + r)
    return sorted(s, key=s.get, reverse=True)
queries = [("gentle mystery set in Kyoto", lambda b: b.subject == "mystery" and "Kyoto" in b.title),
           ("children's book about space", lambda b: b.subject == "children's science" and "space" in b.summary),
           ("ramen recipes", lambda b: b.subject == "cooking" and "ramen" in b.summary),
           ("京都の探偵小説", lambda b: b.subject == "mystery" and "Kyoto" in b.title)]
for name, fn in [("keyword", kw), ("vector", vec), ("hybrid", lambda q: rrf([kw(q), vec(q)]))]:
    per_q = [any(ok(books.iloc[i]) for i in fn(q)[:5]) for q, ok in queries]
    print(f"{name:<8} hit@5 {np.mean(per_q):.2f}  per query: {per_q}")
print("The last query is Japanese for Kyoto detective novel. Offline stand-in embeddings cannot match it to English text. Use a multilingual embedding model and evaluate a Japanese slice.")

## 4. Reservations must be atomic

In [ ]:
conn = sqlite3.connect("data/library.db")
copies.to_sql("copies", conn, if_exists="replace", index=False)
def reserve(copy_id):
    with conn:                                                   # one transaction
        cur = conn.execute("UPDATE copies SET available = 0 WHERE copy_id = ? AND available = 1", (copy_id,))
        return cur.rowcount == 1
cid = copies[copies.available].copy_id.iloc[0]
print("First reservation:", reserve(cid), "| Second reservation of the same copy:", reserve(cid))

## 5. Golden set seed

In [ ]:
golden = [{"id": "book-001", "query": "historical fiction about samurai", "expect_subject": "historical fiction"},
          {"id": "book-002", "query": "something for a 10 year old about space", "expect_subject": "children's science"},
          {"id": "book-003", "query": "京都のミステリー", "expect_subject": "mystery", "lang": "ja"}]
with open("data/golden_seed.jsonl", "w") as f:
    for g in golden: f.write(json.dumps(g, ensure_ascii=False) + "\n")
print("Seeded", len(golden), "golden items, including a Japanese query. Extend to 25.")

## NEW (v4): Modernisation step
Every capstone project includes one modernisation step. Below is a small legacy function from the customer's old system. Run the characterisation tests, then migrate it with OpenAI Codex in small, verified pull requests. Keep these tests green before and after each step.

In [ ]:
# Legacy code to modernise (do not edit it; pin its behaviour first)
import sqlite3

def legacy_import(rows, conn):
    """Old CSV import: builds SQL strings by hand (injection risk) and skips rows silently."""
    conn.execute("CREATE TABLE IF NOT EXISTS books (id TEXT, title TEXT, year INT)")
    n = 0
    for r in rows:
        if not r.get("title"):
            continue
        conn.execute("INSERT INTO books VALUES ('%s', '%s', %d)" % (r["id"], r["title"].replace("'", "''"), int(r["year"])))
        n += 1
    return n

conn = sqlite3.connect(":memory:")
rows = [{"id": "B1", "title": "Kyoto Mystery", "year": 2019}, {"id": "B2", "title": "", "year": 2020},
        {"id": "B3", "title": "Ronin's Path", "year": 2015}]
# Characterisation tests
assert legacy_import(rows, conn) == 2
assert conn.execute("SELECT title FROM books WHERE id='B3'").fetchone()[0] == "Ronin's Path"
print("Characterisation tests pass. Migrate to SQLAlchemy models with parameterised writes.")

---
**Ganbatte** (頑張って): *Do your best*

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)